In [35]:
import requests
import json

# Test search query
url = "http://localhost:8108/collections/category_index/documents/search"
headers = {
    "X-TYPESENSE-API-KEY": "xyz"
}

# Search for a sample value - adjust the query as needed
search_term = "IRON"
search_term = "lenalidomide capsule"
search_term = "omez" 
Search_term = "Omeprazol"# Example search term
search_term ="nimesulde" 
search_term = "atorvastatin"
params = {
    "q": search_term,
    "query_by": "Value",
    "per_page": 10
}

response = requests.get(url, headers=headers, params=params)
data = response.json()
# Pretty-print the result
# print(json.dumps(data, indent=2))


In [36]:
#Total result found
print(f"Total Result foudn:{len(data['hits'])}")
#Printing 7hit resul restul
print(f"Printing KPI for x result:{data['hits'][0]['document']['Value']}")
# Display the matched values in a more readable format
if data.get('found', 0) > 0:
    print("\nMatched Categories:")
    for i, hit in enumerate(data['hits'], 1):
        doc = hit['document']
        print(f"{i}. Value: {doc['Value']}")
        print(f"   Table: {doc['Table_name']}")
        print(f"   Column: {doc['Column_name']}")
        print(f"   Path: {doc['Table_path']}")
        print()
else:
    print("No Categorical match found")

Total Result foudn:10
Printing KPI for x result:Atorvastatin

Matched Categories:
1. Value: Atorvastatin
   Table: Product Master.BPC_PRODUCTGROUP
   Column: BPC_PRODUCTGROUP
   Path: finance-analytics-291816.BI_Sales_Chatbot.Product

2. Value: ATORVASTATIN
   Table: External Sales.MOLECULE
   Column: molecule
   Path: finance-analytics-291816.BI_Sales_Chatbot.VW_FT_FACT_EXTERNAL_SALES

3. Value: ATORVASTATIN
   Table: Daily Sales.Molecule
   Column: Molecule
   Path: finance-analytics-291816.BI_Sales_Chatbot.layer4_FT_VW_FACT_DAILY_SALES_DATA_ANALYSIS

4. Value: Atorvastatin Canada
   Table: Product Master.BPC_PRODUCTGROUP
   Column: BPC_PRODUCTGROUP
   Path: finance-analytics-291816.BI_Sales_Chatbot.Product

5. Value: Atorvastatin 40mg Tab 30's Blister SG
   Table: Product Master.Description
   Column: Description
   Path: finance-analytics-291816.BI_Sales_Chatbot.Product

6. Value: Atorvastatin 20mg Tab 30's Blister SG
   Table: Product Master.Description
   Column: Description
   P

⚙️ Updated CATMatcher Class with Typesense Search


In [44]:
import typesense
import re

class CategoryMatcher:
    def __init__(self, typesense_host="localhost", port="8108", api_key="xyz"):
        self.client = typesense.Client({
            "nodes": [{
                "host": typesense_host,
                "port": port,
                "protocol": "http"
            }],
            "api_key": api_key,
            "connection_timeout_seconds": 2
        })
        # self.stop_words = {'show', 'me', 'the', 'in', 'for', 'of', 'and', 'with', 'by', 'top', 'terms'}


    def find_categories(self, user_input: str, top_k: int = 10):
        try:

            
            # Initialize an empty list to store all matches
            all_matches = []
            
            # Search for each word in the processed text
            search_parameters = {
                    'q': user_input,
                    'query_by': 'Value',
                    'num_typos': 2,
                    'per_page': top_k
                }
                
                # Perform the search for each word
            results = self.client.collections['category_index'].documents.search(search_parameters)
                
            # Extract unique category values, table names, and column names
            for hit in results['hits']:
                doc = hit['document']
                match_info = {
                    'Value': doc['Value'],
                    'Table_name': doc['Table_name'],
                    'Column_name': doc['Column_name'],
                    'Table_path': doc['Table_path']
                }
                all_matches.append(match_info)

            # Convert to a dictionary to remove duplicates
            seen = set()
            unique_matches = []
            for match in all_matches:
                key = (match['Value'], match['Table_name'])
                if key not in seen:
                    seen.add(key)
                    unique_matches.append(match)

            return unique_matches

        except Exception as e:
            print(f"Error querying Typesense: {e}")
            return []


In [47]:
matcher = CategoryMatcher()

search_term = "IRON"
search_term = "lenalidomide capsule"
search_term = "omez" 
Search_term = "Omeprazol"# Example search term
search_term ="nimesulde" 
search_term = "atorvastatin"
results = matcher.find_categories(search_term)

# print("Matched Categorical Values:", matched_kpis)

for i, result in enumerate(results, 1):
    print(f"{i}. Value: {result['Value']}")
    print(f"   Table: {result['Table_name']}")
    print(f"   Column: {result['Column_name']}")
    print(f"   Path: {result['Table_path']}")
    print()


1. Value: Atorvastatin
   Table: Product Master.BPC_PRODUCTGROUP
   Column: BPC_PRODUCTGROUP
   Path: finance-analytics-291816.BI_Sales_Chatbot.Product

2. Value: ATORVASTATIN
   Table: External Sales.MOLECULE
   Column: molecule
   Path: finance-analytics-291816.BI_Sales_Chatbot.VW_FT_FACT_EXTERNAL_SALES

3. Value: ATORVASTATIN
   Table: Daily Sales.Molecule
   Column: Molecule
   Path: finance-analytics-291816.BI_Sales_Chatbot.layer4_FT_VW_FACT_DAILY_SALES_DATA_ANALYSIS

4. Value: Atorvastatin Canada
   Table: Product Master.BPC_PRODUCTGROUP
   Column: BPC_PRODUCTGROUP
   Path: finance-analytics-291816.BI_Sales_Chatbot.Product

5. Value: Atorvastatin 40mg Tab 30's Blister SG
   Table: Product Master.Description
   Column: Description
   Path: finance-analytics-291816.BI_Sales_Chatbot.Product

6. Value: Atorvastatin 20mg Tab 30's Blister SG
   Table: Product Master.Description
   Column: Description
   Path: finance-analytics-291816.BI_Sales_Chatbot.Product

7. Value: Aspirin (75mg) +

In [43]:
len(matched_kpis  )

30